# Step 1, session B of 2 (~10.8 h on a T4)

Mamba final runs (5 flows, seeds 0, 1, 2, one run per seed) with the smoothness weight chosen in session A, then the fully observed Mamba ceiling (skipped if time runs out). **Set `LAMBDA_MAMBA` to the value session A printed**, then Run all.

In [ ]:
REPO_URL = "https://github.com/muhammadhussainqureshi/thesis-gs"
CONFIG = "thesis1_step1_b"
BUDGET_HOURS = 11.3
LAMBDA_MAMBA = None    # <-- paste the value printed at the end of session A, e.g. 1.0
assert LAMBDA_MAMBA is not None, "set LAMBDA_MAMBA from session A"

import os, subprocess, sys, glob, json, shutil, torch
WORK = "/kaggle/working"
CODE = "/kaggle/working/thesis-gs"
DATA = f"{WORK}/data"
OUT = f"{WORK}/{CONFIG}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.isfile(f"{CODE}/configs/{CONFIG}.json"), "config not found - is the latest code pushed?"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
cfg = json.load(open(f"{CODE}/configs/{CONFIG}.json"))
for r in cfg["runs"]:
    r["args"] = [str(LAMBDA_MAMBA) if x == "LAMBDA_MAMBA" else x for x in r["args"]]
cfg_path = f"{WORK}/{CONFIG}_used.json"
json.dump(cfg, open(cfg_path, "w"), indent=1)
cmd = [sys.executable, f"{CODE}/bin/run_experiments.py", cfg_path, "--out", OUT, "--data_dir", DATA,
       "--budget_hours", str(BUDGET_HOURS), "--zip", f"{WORK}/{CONFIG}_results.zip"]
subprocess.run(cmd)        # failures are logged per run; the notebook keeps going


In [ ]:
from IPython.display import Image, Markdown, display
subprocess.run([sys.executable, f"{CODE}/bin/ns4d_step1_report.py", OUT, "--out", f"{OUT}/step1_report"])
for f in ["per_frame.png", "spectra.png"]:
    if os.path.exists(f"{OUT}/step1_report/{f}"):
        display(Image(filename=f"{OUT}/step1_report/{f}"))
for f in glob.glob(f"{OUT}/**/*.pt", recursive=True):   # keep the zip small
    os.remove(f)
shutil.make_archive(f"{WORK}/{CONFIG}_results", "zip", OUT)
print(f"Download {CONFIG}_results.zip from the Output tab.")
